# Deploy Product Agent to Amazon Bedrock AgentCore Runtime

This tutorial series builds a **5-agent e-commerce assistant** using **Strands Agents GraphBuilder** for deterministic routing across Amazon Bedrock AgentCore runtimes. In this notebook, you deploy the Product Agent -- an A2A server that searches product catalogs via external HTTP API.

**Notebook 2 of 5** -- The Graph Orchestrator routes to this agent for BROWSE and RECOMMEND intents.

## Architecture Overview

This tutorial deploys 5 agents across 5 Amazon Bedrock AgentCore runtimes. The Product Agent (highlighted below) handles product catalog queries:

| Runtime | Agent | Protocol | Tools |
|---------|-------|----------|-------|
| 1 | Classifier | A2A (port 9000) | None (pure LLM) |
| **2** | **Product** | **A2A (port 9000)** | **HTTP API tools** |
| 3 | Order | A2A (port 9000) | DynamoDB via MCP |
| 4 | Recommendation | A2A (port 9000) | None (LLM synthesis) |
| 5 | Graph Orchestrator | HTTP (port 8080) | GraphBuilder + A2A clients |

The Product Agent is invoked for:
- **BROWSE** intent: Direct product search from classifier
- **RECOMMEND** intent: Provides catalog data to Recommendation Agent (runs in parallel with Order Agent)

## Prerequisites

- [AWS CLI](https://aws.amazon.com/cli/) installed and configured
- Python 3.10 or higher
- Docker or Podman installed (only for local container builds; not required if using CodeBuild)
- Claude Sonnet 4 model access in Amazon Bedrock
- **Notebook 01 completed**: Classifier Agent deployed and URL stored in SSM

In [ ]:
import os
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

import boto3

NOTEBOOK_DIR = Path.cwd()

from utils import (
    PRODUCT_AGENT_NAME,
    PRODUCT_ROLE_NAME,
    SSM_PRODUCT_AGENT_URL,
    create_agentcore_role,
    store_agent_url,
)

session = boto3.Session()
region = session.region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Agent Name: {PRODUCT_AGENT_NAME}")

---
## Step 1: Create Product Agent with A2A Protocol

The Product Agent uses three custom `@tool` functions to query the DummyJSON product catalog API:

| Tool | Endpoint | Purpose |
|------|----------|--------|
| `search_products` | `/products/search?q=` | Keyword search across all categories |
| `get_products_by_category` | `/products/category/` | Browse specific product categories |
| `get_all_products` | `/products` | List all available products |

### Code Structure

The following cell creates `product_agent/a2a_server.py`:

| Section | What It Does |
|---------|--------------|
| Product Catalog Tools | Three `@tool` functions for search, category browse, and listing |
| Callback Handler | Logs tool invocations to CloudWatch for debugging |
| Agent Creation | Agent with 3 tools and product-focused system prompt |
| A2A Server | Wraps agent for inter-agent communication on port 9000 |

In [ ]:
%%writefile product_agent/a2a_server.py
"""Product Agent deployed to Amazon Bedrock AgentCore with A2A protocol support.

Queries product catalog from DummyJSON API (https://dummyjson.com) using custom
tools built with the Strands @tool decorator.

Key features:
- A2A protocol server for graph-based multi-agent orchestration via Agent-to-Agent messaging
- Custom HTTP request tools for external API access
- OpenTelemetry instrumentation for AWS X-Ray distributed tracing
"""

import json
import logging
import os
from typing import Any

import boto3
import requests
import uvicorn
from fastapi import FastAPI
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer
from strands.telemetry import StrandsTelemetry

logging.basicConfig(level=logging.INFO)
logging.getLogger("strands").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

StrandsTelemetry().setup_otlp_exporter()

PORT = 9000
SSM_PRODUCT_AGENT_URL = "/ecommerce-graph/product-agent-url"

SYSTEM_PROMPT = """You are a Product Agent for an e-commerce assistant.

You have access to tools that query a product catalog with 194+ products across multiple categories.

Available tools:
- search_products: Search for products by keyword (e.g., "laptop", "phone")
- get_products_by_category: Get products from specific categories
- get_all_products: Browse all available products

Electronics categories available:
- laptops (MacBook Pro, Dell XPS, ThinkPad, etc.)
- smartphones (iPhone, Samsung, Google Pixel, etc.)
- tablets (iPad, Samsung Tab, etc.)
- mobile-accessories (phone cases, chargers, etc.)

Other categories: beauty, fragrances, furniture, groceries, mens-shirts, womens-dresses, and more.

Help customers find products by searching, browsing categories, or filtering by price.
If a product isn't found, suggest similar alternatives from available categories.
"""

@tool
def search_products(query: str, limit: int = 10) -> str:
    """Search for products by keyword across all categories.

    Args:
        query: Search term (e.g., "laptop", "phone", "MacBook")
        limit: Maximum number of products to return (default: 10)

    Returns:
        JSON string with matching products including id, title, price, category, description
    """
    try:
        url = f"https://dummyjson.com/products/search?q={query}&limit={limit}"
        response = requests.get(url, timeout=10)
        data = response.json()
        products = data.get("products", [])
        result = []
        for p in products:
            result.append({
                "id": p["id"],
                "title": p["title"],
                "price": p["price"],
                "category": p["category"],
                "description": p.get("description", ""),
                "rating": p.get("rating", 0),
            })
        return json.dumps({"products": result, "total": data.get("total", 0)})
    except Exception as e:
        logger.error(f"Error searching products: {e}")
        return json.dumps({"error": str(e)})


@tool
def get_products_by_category(category: str, limit: int = 10) -> str:
    """Get products from a specific category.

    Args:
        category: Category name (e.g., "laptops", "smartphones", "beauty")
        limit: Maximum number of products to return (default: 10)

    Returns:
        JSON string with products in the specified category
    """
    try:
        url = f"https://dummyjson.com/products/category/{category}?limit={limit}"
        response = requests.get(url, timeout=10)
        data = response.json()
        products = data.get("products", [])
        result = []
        for p in products:
            result.append({
                "id": p["id"],
                "title": p["title"],
                "price": p["price"],
                "category": p["category"],
                "description": p.get("description", ""),
                "rating": p.get("rating", 0),
            })
        return json.dumps({"products": result, "total": len(result)})
    except Exception as e:
        logger.error(f"Error getting category {category}: {e}")
        return json.dumps({"error": str(e)})


@tool
def get_all_products(limit: int = 30) -> str:
    """Browse all available products in the catalog.

    Args:
        limit: Maximum number of products to return (default: 30)

    Returns:
        JSON string with product listings
    """
    try:
        url = f"https://dummyjson.com/products?limit={limit}"
        response = requests.get(url, timeout=10)
        data = response.json()
        products = data.get("products", [])
        result = []
        for p in products:
            result.append({
                "id": p["id"],
                "title": p["title"],
                "price": p["price"],
                "category": p["category"],
                "rating": p.get("rating", 0),
            })
        return json.dumps({"products": result, "total": data.get("total", 0)})
    except Exception as e:
        logger.error(f"Error getting products: {e}")
        return json.dumps({"error": str(e)})


class ToolLoggingHandler:
    def __init__(self):
        self.logged_tool_ids = set()
        self.tool_count = 0
    def __call__(self, **kwargs):
        message = kwargs.get("message", {})
        if isinstance(message, dict) and message.get("role") == "assistant":
            for content in message.get("content", []):
                if isinstance(content, dict):
                    tool_use = content.get("toolUse")
                    if tool_use:
                        tool_id = tool_use.get("toolUseId")
                        if tool_id and tool_id not in self.logged_tool_ids:
                            self.logged_tool_ids.add(tool_id)
                            self.tool_count += 1
                            logger.info(f"=== TOOL #{self.tool_count}: {tool_use.get('name', 'Unknown')} ===")
                            input_str = json.dumps(tool_use.get("input", {}))
                            if len(input_str) > 2000:
                                input_str = input_str[:2000] + "..."
                            logger.info(f"TOOL INPUT: {input_str}")
        if kwargs.get("complete") and kwargs.get("data"):
            logger.info(f"=== COMPLETE: {len(kwargs.get('data', ''))} chars ===")


# Get AWS region
session = boto3.Session()
region = session.region_name or os.environ.get("AWS_REGION", "us-west-2")

product_agent = Agent(
    name="Ecommerce_Graph_Product",
    description="Product catalog search agent for e-commerce assistant",
    system_prompt=SYSTEM_PROMPT,
    model=BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
        region_name=region,
    ),
    tools=[search_products, get_products_by_category, get_all_products],
    callback_handler=ToolLoggingHandler(),
)

app = FastAPI()

@app.get("/ping")
async def health():
    return {"status": "healthy"}

a2a_server = A2AServer(agent=product_agent, serve_at_root=True)
a2a_server.setup(app)

if __name__ == "__main__":
    logger.info(f"Starting Product Agent on port {PORT}")
    uvicorn.run(app, host="0.0.0.0", port=PORT)

In [ ]:
%%writefile product_agent/requirements.txt
strands-agents[a2a,otel]
fastapi
uvicorn
boto3
requests

---
## Step 2: Deploy to Amazon Bedrock AgentCore

The `bedrock-agentcore-starter-toolkit` handles the deployment pipeline:

1. **Creates IAM role** -- Grants permissions for ECR image pull, Bedrock model invocation, and CloudWatch logging
2. **Configures runtime** -- Packages agent code into a deployable container configuration
3. **Launches runtime** -- Builds Docker image, pushes to ECR, and creates the AgentCore runtime

### Create IAM Role and Configure Runtime

The IAM execution role allows the AgentCore runtime to pull container images, invoke Bedrock models, and write logs.

| Parameter | Value | Purpose |
|-----------|-------|--------|
| `entrypoint` | `a2a_server.py` | Python file that starts the A2A server |
| `protocol` | `A2A` | Enables Agent-to-Agent communication on port 9000 |
| `agent_name` | `ecommerce_graph_product` | Unique identifier for this runtime |

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

product_role_arn = create_agentcore_role(PRODUCT_ROLE_NAME, account_id, region)
print(f"IAM Role ARN: {product_role_arn}")

product_agent_dir = NOTEBOOK_DIR / "product_agent"
os.chdir(product_agent_dir)

product_runtime = Runtime()
product_runtime.configure(
    entrypoint="a2a_server.py",
    execution_role=product_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=PRODUCT_AGENT_NAME,
    protocol="A2A",
)

os.chdir(NOTEBOOK_DIR)
print(f"Runtime configured: {PRODUCT_AGENT_NAME}")

### Fix Dockerfile Permissions

AgentCore containers run as the `bedrock_agentcore` user, not root. The auto-generated Dockerfile uses `COPY . .` which preserves host file ownership, causing permission errors. This cell updates it to `COPY --chown=bedrock_agentcore:bedrock_agentcore . .`.

In [ ]:
# # Fix Dockerfile COPY ownership
# dockerfile_path = "Dockerfile"
# with open(dockerfile_path, 'r') as f:
#     content = f.read()
# if 'COPY --chown=bedrock_agentcore:bedrock_agentcore . .' not in content:
#     content = content.replace('COPY . .', 'COPY --chown=bedrock_agentcore:bedrock_agentcore . .')
#     with open(dockerfile_path, 'w') as f:
#         f.write(content)
#     print("Dockerfile updated with correct ownership")
# else:
#     print("Dockerfile already has correct ownership")

### Launch Agent

`Runtime.launch()` builds the Docker image, pushes it to ECR, and creates the AgentCore runtime.

**Note:** First deployment takes 5-10 minutes (subsequent updates are faster).

In [ ]:
print("Launching Product Agent (this may take several minutes)...")
os.chdir(product_agent_dir)
product_launch = product_runtime.launch(auto_update_on_conflict=True)
print(f"Product Agent ARN: {product_launch.agent_arn}")
PRODUCT_AGENT_ARN = product_launch.agent_arn
os.chdir(NOTEBOOK_DIR)

### Get Runtime URL

Once the runtime reaches `ACTIVE` or `READY` status, construct the invocation URL from the runtime ARN.

In [ ]:
os.chdir(product_agent_dir)
status_response = product_runtime.status()
status = status_response.endpoint.get("status", "")
product_agent_url = None

print(f"Product Agent Status: {status}")

if status.upper() in ["ACTIVE", "READY"]:
    agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn")
    escaped_arn = quote(agent_runtime_arn, safe="")
    product_agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    print(f"Runtime URL: {product_agent_url}")
else:
    print(f"Agent not ready. Current status: {status}")

os.chdir(NOTEBOOK_DIR)

### Store Runtime URL in Parameter Store

Store the URL in AWS Systems Manager Parameter Store so other components can discover this agent:
- **Agent container** reads it at startup to populate the agent card's `http_url` field
- **Graph Orchestrator** (Notebook 5) retrieves it to configure `A2AClientToolProvider` for the product node

In [ ]:
if status.upper() in ["ACTIVE", "READY"]:
    result = store_agent_url(
        param_name=SSM_PRODUCT_AGENT_URL,
        url=product_agent_url,
        region=region,
    )
    print(result["message"])
    print(f"Parameter version: {result['version']}")
else:
    print("Skipping SSM storage - agent not ready")

In [ ]:
print("=" * 60)
print("Product Agent Deployment Summary")
print("=" * 60)
print(f"Agent Name: {PRODUCT_AGENT_NAME}")
print(f"Agent ARN: {PRODUCT_AGENT_ARN}")
print(f"IAM Role: {product_role_arn}")
print(f"Runtime URL: {product_agent_url or 'Not available - agent not ready'}")
print(f"SSM Parameter: {SSM_PRODUCT_AGENT_URL}")
print("=" * 60)

---
## Step 3: Test Product Agent

Verify the agent works standalone before integrating with the Graph Orchestrator.

**Example queries to try:**
- `"Show me electronics under $100"` -- keyword search
- `"What laptops do you have?"` -- category search
- `"Browse all products"` -- full catalog listing

In [ ]:
test_message = "Show me electronics under $100"

payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": str(uuid4()),
    "params": {
        "message": {
            "messageId": str(uuid4()),
            "role": "user",
            "parts": [{"kind": "text", "text": test_message}],
        }
    },
}

os.chdir(product_agent_dir)
print(f"Testing: '{test_message}'")
print("-" * 40)
response = product_runtime.invoke(payload, session_id=str(uuid4()))
print(f"\nResponse:\n{response}")
os.chdir(NOTEBOOK_DIR)

### Verify Agent Card (A2A Discovery)

The `A2AServer` wrapper automatically generates an A2A-compliant agent card from your Strands agent.

In [ ]:
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests
import json

agent_card_url = f"{product_agent_url}/.well-known/agent-card.json"
credentials = boto3.Session().get_credentials()
request = AWSRequest(method="GET", url=agent_card_url)
SigV4Auth(credentials, "bedrock-agentcore", region).add_auth(request)

response = requests.get(
    agent_card_url,
    headers=dict(request.headers),
)
print(f"Status: {response.status_code}")
if response.status_code == 200:
    card = response.json()
    print(json.dumps(card, indent=2))

---
## Next Steps

| Notebook | What You'll Build |
|----------|-------------------|
| **3. Deploy Order Agent** | DynamoDB integration with MCP tools and async lifespan pattern |
| **4. Deploy Recommendation Agent** | LLM synthesis agent for personalized recommendations |
| **5. Deploy Graph Orchestrator** | GraphBuilder DAG that routes through all agents with conditional edges |

---
## Cleanup (Optional)

Run this section to delete all resources created by this notebook.

In [ ]:
print("Destroying Product Agent...")
os.chdir(product_agent_dir)
try:
    product_runtime.destroy(delete_ecr_repo=True)
    print("Product Agent destroyed")
except Exception as e:
    print(f"Error: {e}")
os.chdir(NOTEBOOK_DIR)

In [ ]:
ssm = boto3.client("ssm", region_name=region)
try:
    ssm.delete_parameter(Name=SSM_PRODUCT_AGENT_URL)
    print(f"Deleted SSM parameter: {SSM_PRODUCT_AGENT_URL}")
except Exception as e:
    print(f"Error deleting SSM parameter: {e}")

print("Cleaning up auto-generated files...")
for cleanup_file in ["Dockerfile", ".dockerignore"]:
    cleanup_path = product_agent_dir / cleanup_file
    if cleanup_path.exists():
        cleanup_path.unlink()
        print(f"  Deleted: {cleanup_file}")